In [82]:
import os
import re
import json
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd

In [83]:
def find_project_root(marker=".git"):
    """Find the root directory of the project based on a marker file or directory."""
    path = Path(os.getcwd()).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    return path


PROJECT_ROOT = find_project_root()
BASE_DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = BASE_DATA_DIR / "raw"
PROCESSED_DIR = BASE_DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [84]:
path = kagglehub.dataset_download("moesiof/portuguese-narrative-essays")
print("Path to dataset files:", path)

Path to dataset files: /Users/hyan/.cache/kagglehub/datasets/moesiof/portuguese-narrative-essays/versions/1


In [85]:
for file in Path(path).glob("**/*.csv"):
    print(file)

/Users/hyan/.cache/kagglehub/datasets/moesiof/portuguese-narrative-essays/versions/1/validation.csv
/Users/hyan/.cache/kagglehub/datasets/moesiof/portuguese-narrative-essays/versions/1/test.csv
/Users/hyan/.cache/kagglehub/datasets/moesiof/portuguese-narrative-essays/versions/1/train.csv


In [86]:
def load_dataset(path: str, file_name: str):
    """
    Loads the dataset from the given path.
    """
    csv_path = os.path.join(path, file_name)
    return pd.read_csv(csv_path)

train = load_dataset(path, "train.csv")
val = load_dataset(path, "validation.csv")
test = load_dataset(path, "test.csv")

In [87]:
train = train.drop(columns='prompt')
test = test.drop(columns='prompt')
val = val.drop(columns='prompt')

In [88]:
def get_tags(essays: pd.Series) -> np.ndarray:
    """Utility function to list all tags in
    a series of essays.
    """
    flags = []
    for e in essays:
        square = re.findall(r"(\[.{1,3}\])", e)
        curly = re.findall(r"(\{.{1,3}\})", e)
        mixed_1 = re.findall(r"(\[.{1,3}\})", e)
        mixed_2 = re.findall(r"(\{.{1,3}\])", e)

        for group in [square, curly, mixed_1, mixed_2]:
            flags.extend([g for g in group])

    return np.sort(np.unique(flags))


def get_tag_regex() -> str:
    """Returns the regex for the tags."""
    # Well-formed tags with format [<LETTER_OR_SYMBOL>]
    tag_regex = r"(\[[PpSsTtXx?]\])"

    # Well-formed tags with format {<LETTER_OR_SYMBOL>}
    tag_regex += r"|({[ptx?]})"

    # Well-formed tags [LT] or [LC]
    tag_regex += r"|(\[L[TC]\])"

    # Well-formed tags with format [lt] or [lc]
    tag_regex += r"|(\[l[tc]\])"

    # Variant with a trailing space
    tag_regex += r"|(\[ P\])"

    # Mixed closing/opening symbol
    tag_regex += r"|(\[[PX?]\})"
    tag_regex += r"|(\{?\])"
    return tag_regex


def remove_tags(df: pd.DataFrame, tag_regex: str) -> pd.DataFrame:
    """Removes tags from the essay column of a dataframe."""
    df = df.copy()
    df["essay"] = df["essay"].str.replace(tag_regex, "", regex=True)
    return df

In [89]:
np.concatenate([get_tags(train.essay), get_tags(test.essay), get_tags(val.essay)])

array(['[ P]', '[?]', '[?}', '[LC]', '[LT]', '[P]', '[P}', '[S]', '[T]',
       '[X]', '[X}', '[X~]', '[lt]', '[p]', '[s]', '[t]', '[x]', '{?]',
       '{?}', '{p}', '{t}', '[?]', '[LC]', '[LT]', '[P]', '[R]', '[S]',
       '[T]', '[X]', '[X}', '[p]', '[s]', '[t]', '[x]', '{?}', '{p}',
       '{t}', '{x}', '[?]', '[LC]', '[LT]', '[P]', '[R]', '[S]', '[T]',
       '[X]', '[p]', '[r]', '[s]', '[t]', '[x]'], dtype='<U4')

In [90]:
tag_regex = get_tag_regex()
train = remove_tags(train, tag_regex)
test = remove_tags(test, tag_regex)
val = remove_tags(val, tag_regex)

In [91]:
np.concatenate([get_tags(train.essay), get_tags(test.essay), get_tags(val.essay)])

array([], dtype=float64)

In [92]:
train.head()

,id,essay,formal_register,thematic_coherence,narrative_rhetorical_structure,cohesion
0,753,Os vidros de tintas\nOs vridos de tintas pint...,3,1,1,1
1,582,O ARMÁRIO E TINTA MAGICA\nEU ENCONTREI CON TI...,2,2,3,2
2,548,Li um livro que me assustou !\n Em uma (belea...,3,1,5,3
3,113,Um dia eu vi mas latas de tinta em de \nai res...,3,3,5,3
4,174,era uma vez uma menina Que esta uque faze\ne ...,2,2,4,2


In [93]:
train.to_csv(RAW_DIR / "train.csv", index=False)
test.to_csv(RAW_DIR / "test.csv", index=False)
val.to_csv(RAW_DIR / "validation.csv", index=False)

In [94]:
def to_jsonl(df: pd.DataFrame, file_path: str):
    """Converts a dataframe to a jsonl file."""
    instruction = "Você é um especialista no ensino de redação narrativa para alunos do ensino fundamental. Dada uma determinada entrada (uma redação narrativa), seu papel é avaliá-la nas seguintes competências: registro formal, coerência temática, estrutura retórica narrativa e coesão. Sua nota deve ir de 1 à 5, sendo 1 a menor nota e 5 a maior. A saída deve ser um dicionário com as notas de cada competência, por exemplo: {\"formal_register\": 5, \"themetic_coherence\": 2, \"narrative_rhetorical_structure\": 4, \"cohesion\": 3}"
    
    records = []
    for _, row in df.iterrows():
        output_dict = {
            "formal_register": row["formal_register"],
            "thematic_coherence": row["thematic_coherence"],
            "narrative_rhetorical_structure": row["narrative_rhetorical_structure"],
            "cohesion": row["cohesion"],
        }
        
        records.append({
            "instruction": instruction,
            "input": row["essay"],
            "output": json.dumps(output_dict),
            "answer": json.dumps(output_dict),
        })
        
    with open(file_path, 'w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record) + '\n')

In [95]:
to_jsonl(train, str(Path(PROCESSED_DIR / "train.jsonl")))
to_jsonl(val, str(Path(PROCESSED_DIR / "val.jsonl")))
to_jsonl(test, str(Path(PROCESSED_DIR / "test.jsonl")))